## **For this notebook to work, you must execute ALL the cells in the order in which they appear.**

In [6]:
!wget -O 2019.csv https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/2019.csv

--2026-11-14 13:47:43--  https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/2019.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 123882 (121K) [text/plain]
Saving to: ‘2019.csv’

2019.csv            100%[===================>] 120.98K  --.-KB/s    in 0.05s   

2026-11-14 13:47:43 (2.44 MB/s) - ‘2019.csv’ saved [123882/123882]



In [7]:
# 2. Inicializar Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Colab_Session_Name") \
    .getOrCreate()

# **Upload a CSV that does not have column headers**

In [8]:
# 3. readCSV using local  path
df = spark.read.format("csv").option("header", "true").load("2019.csv")

# 4. Display DF
df.show(5)             # in fabric use display(df)

+-------+---+----------+--------------+-----------------------------+-----------------------+---+--------+--------+
|SO43701| 11|2019-07-01|   Christy Zhu|christy12@adventure-works.com|Mountain-100 Silver, 44| 16| 3399.99|271.9992|
+-------+---+----------+--------------+-----------------------------+-----------------------+---+--------+--------+
|SO43704|  1|2019-07-01|    Julio Ruiz|         julio1@adventure-...|   Mountain-100 Blac...|  1| 3374.99|269.9992|
|SO43705|  1|2019-07-01|     Curtis Lu|         curtis9@adventure...|   Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43700|  1|2019-07-01|  Ruben Prasad|         ruben10@adventure...|     Road-650 Black, 62|  1|699.0982| 55.9279|
|SO43703|  1|2019-07-01|Albert Alvarez|         albert7@adventure...|       Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43697|  1|2019-07-01|   Cole Watson|         cole1@adventure-w...|       Road-150 Red, 62|  1| 3578.27|286.2616|
+-------+---+----------+--------------+-----------------------------+---

# **option("header","false") will indicate that the CSV does not contain column headers**

In [9]:
# Tell Spark that the CSV does not have column headers. It will temporarily replace them with C0, C1, C2, etc.
df = spark.read.format("csv").option("header","false").load("2019.csv")
df.show()     # In Fabric use:   display(df)

+-------+---+----------+-----------------+--------------------+--------------------+---+--------+--------+
|    _c0|_c1|       _c2|              _c3|                 _c4|                 _c5|_c6|     _c7|     _c8|
+-------+---+----------+-----------------+--------------------+--------------------+---+--------+--------+
|SO43701|  1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43704|  1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|  1| 3374.99|269.9992|
|SO43705|  1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|  1| 3399.99|271.9992|
|SO43700|  1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|  1|699.0982| 55.9279|
|SO43703|  1|2019-07-01|   Albert Alvarez|albert7@adventure...|    Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43697|  1|2019-07-01|      Cole Watson|cole1@adventure-w...|    Road-150 Red, 62|  1| 3578.27|286.2616|
|SO43699|  1|2019-07-01|    Sydney Wr

## **Assign column headers to the DataFrame**

In [11]:
# assign column headers to the DataFrame
from pyspark.sql.types import *

orderSchema = StructType([
   StructField("SalesOrderNumber", StringType()),
   StructField("SalesOrderLineNumber", IntegerType()),
   StructField("OrderDate", DateType()),
   StructField("CustomerName", StringType()),
   StructField("Email", StringType()),
   StructField("Item", StringType()),
   StructField("Quantity", IntegerType()),
   StructField("UnitPrice", FloatType()),
   StructField("Tax", FloatType())
])

df = spark.read.format("csv").schema(orderSchema).load("2019.csv")
df.show()     # In Fabric, use display(df)

+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         

In [14]:
# 1. Convert to Pandas and save in Colab (this file is lost when the Colab session ends)
df_pandas = df.toPandas()
df_pandas.to_csv("exported_file.csv", index=False)

# 2. Download the CSV file to your computer
from google.colab import files
files.download("exported_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Save parquet file and retrieve them when needed.**

In [16]:
# 1. Save the DF as a Parquet file in Colab’s local temporary directory.
# In Colab, this file disappears when the session ends, but in Fabric,
# the created folder and the Parquet file remain.
df.write.format("parquet").mode("overwrite").save("orders_parquet")

# 2. Leerlo de vuelta desde la ruta local temporal
df_parquet = spark.read.format("parquet").load("orders_parquet")

# 3. Mostrar el resultado
df_parquet.show(5)

+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|  CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+--------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|   Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|    Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|     Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|  Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|                   1|2019-07-0